<a id="orthogonality"></a>

$$
\text{三角函数的正交性 (Orthogonality of Sine and Cosine)} \\
\text{周期 T, 基频 } \omega_0 = \frac{2\pi}{T} \text{, m, n 为正整数, 积分区间是一个周期 } [0, T] \\ \quad \\
\text{1. 先算 } \int_0^T \cos(k\omega_0 t) \, dt = \left[\frac{\sin(k\omega_0 t)}{k\omega_0}\right]_0^T = \frac{\sin(2\pi k)}{k\omega_0} = 0 \ (k \neq 0) \text{, 同理 } \int_0^T \sin(k\omega_0 t) \, dt = 0 \\
\text{2. 积化和差 (由 } \cos(a \pm b) \text{ 展开相加减得到):} \\
\cos a \cos b = \tfrac{1}{2}[\cos(a - b) + \cos(a + b)], \quad \sin a \sin b = \tfrac{1}{2}[\cos(a - b) - \cos(a + b)], \quad \sin a \cos b = \tfrac{1}{2}[\sin(a + b) + \sin(a - b)] \\
\text{3. 代入 } a = m\omega_0 t, \ b = n\omega_0 t \text{, 由第 1 步, 只有 } (m - n)\omega_0 t = 0 \text{ 的那一项积分不为 0:} \\
\int_0^T \cos(m\omega_0 t)\cos(n\omega_0 t) \, dt = \int_0^T \sin(m\omega_0 t)\sin(n\omega_0 t) \, dt = \begin{cases} 0 & m \neq n \\ \frac{T}{2} & m = n \end{cases}, \qquad \int_0^T \sin(m\omega_0 t)\cos(n\omega_0 t) \, dt = 0 \\ \quad \\
\text{把 } \langle f, g \rangle = \int_0^T f(t) g(t) \, dt \text{ 看成函数的内积, 这些三角函数就是两两正交的 "向量"}
$$

数学基础: [定积分](../math_basics/calculus.ipynb#integral) · [正交矩阵](../math_basics/linear_algebra.ipynb#orthogonal) · 应用: [积化和差与平均功率](AC_DC.ipynb#ac-power)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad
import time

np.set_printoptions(linewidth=np.inf)
np.set_printoptions(suppress=True)

def plot_nicely():
    for ax in plt.gcf().get_axes():
        ax.grid(True, which="both", alpha=0.3)
        if ax.get_legend_handles_labels()[0]:
            ax.legend(fontsize='small')
    plt.tight_layout()

T = 3.0
w0 = 2 * np.pi / T
basis = {f'cos{n}': (lambda t, n=n: np.cos(n * w0 * t)) for n in range(1, 4)}
basis.update({f'sin{n}': (lambda t, n=n: np.sin(n * w0 * t)) for n in range(1, 4)})
names = list(basis)
gram = np.array([[quad(lambda t: basis[a](t) * basis[b](t), 0, T, limit=200)[0] for b in names] for a in names])
print(f'基函数: {names}')
print(f'内积矩阵 (Gram matrix) = \n{np.round(gram, 10) + 0.0}')
print(f'== (T/2) I ? {np.allclose(gram, T / 2 * np.eye(len(names)))}')

<a id="fourier-series"></a>

$$
\text{傅里叶级数 (Fourier Series): 周期函数 = 直流 + 各次谐波的叠加} \\
f(t) = \frac{a_0}{2} + \sum_{n=1}^{\infty} \left[a_n \cos(n\omega_0 t) + b_n \sin(n\omega_0 t)\right] \\ \quad \\
\text{求系数: 两边同乘 } \cos(m\omega_0 t) \text{ 并在一个周期上积分, 由正交性右边只剩 n = m 那一项} \\
\int_0^T f(t) \cos(m\omega_0 t) \, dt = a_m \cdot \frac{T}{2} \implies a_m = \frac{2}{T} \int_0^T f(t) \cos(m\omega_0 t) \, dt \\
\text{同理 } b_m = \frac{2}{T} \int_0^T f(t) \sin(m\omega_0 t) \, dt \text{; m = 0 时 } \int_0^T f \, dt = \frac{a_0}{2} T \text{, 所以 } \frac{a_0}{2} = \frac{1}{T}\int_0^T f \, dt \text{ (平均值 = 直流分量)} \\ \quad \\
\text{几何意义: 和正交基下求坐标一样, 坐标 = 内积 / 基向量长度的平方 } a_m = \frac{\langle f, \cos m\omega_0 t \rangle}{\langle \cos m\omega_0 t, \cos m\omega_0 t \rangle} \\
\text{截断到 N 项的部分和 = f 在前 N 个基函数张成的子空间上的正交投影, 是最小二乘意义下最好的近似}
$$

数学基础: [投影矩阵与最小二乘](../math_basics/linear_algebra.ipynb#projection) · [QR 分解与 Gram-Schmidt 正交化](../math_basics/linear_algebra.ipynb#qr)

<a id="square-wave"></a>

$$
\text{例: 方波 (Square Wave), 一个周期内 } f(t) = +1 \ (0 < t < \tfrac{T}{2}), \ -1 \ (\tfrac{T}{2} < t < T) \\
\text{f 是奇函数 (关于 t = 0 对称取反), 奇函数乘 cos (偶函数) 积分为 0} \implies a_n = 0 \\
b_n = \frac{2}{T}\left[\int_0^{T/2} \sin(n\omega_0 t) \, dt - \int_{T/2}^{T} \sin(n\omega_0 t) \, dt\right] \\
\int_0^{T/2} \sin(n\omega_0 t) \, dt = \frac{1 - \cos(n\pi)}{n\omega_0}, \quad \int_{T/2}^{T} \sin(n\omega_0 t) \, dt = \frac{\cos(n\pi) - 1}{n\omega_0} \quad (\omega_0 \tfrac{T}{2} = \pi, \ \omega_0 T = 2\pi) \\
b_n = \frac{2}{T} \cdot \frac{2(1 - \cos n\pi)}{n\omega_0} = \frac{2(1 - (-1)^n)}{n\pi} = \begin{cases} \frac{4}{n\pi} & \text{n 奇数} \\ 0 & \text{n 偶数} \end{cases} \\
f(t) = \frac{4}{\pi}\left[\sin\omega_0 t + \frac{1}{3}\sin 3\omega_0 t + \frac{1}{5}\sin 5\omega_0 t + \cdots\right] \\ \quad \\
\text{吉布斯现象 (Gibbs Phenomenon): 在跳变点附近, 部分和总会冲过头约 9\% 的跳变幅度 (跳变 = 2, 峰值} \to 1.179 \text{)} \\
\text{项数增加时冲过头的位置向跳变点靠近, 但峰值不会减小 (逐点收敛但不是一致收敛)}
$$

In [ ]:
square = lambda t: np.where((t % T) < T / 2, 1.0, -1.0)

def fourier_coeffs(f, n_max, T, n_samples=200000):
    # 用梯形法数值积分求系数 (端点处的跳变对积分影响很小)
    t = np.linspace(0, T, n_samples + 1)
    w0 = 2 * np.pi / T
    a = np.array([2 / T * np.trapezoid(f(t) * np.cos(n * w0 * t), t) for n in range(n_max + 1)])
    b = np.array([2 / T * np.trapezoid(f(t) * np.sin(n * w0 * t), t) for n in range(n_max + 1)])
    return a, b

a_num, b_num = fourier_coeffs(square, 9, T)
n = np.arange(10)
b_formula = np.where(n % 2 == 1, 4 / (np.pi * np.maximum(n, 1)), 0.0)
print(f'a_n 数值 = {np.round(a_num, 5)}')
print(f'b_n 数值 = {np.round(b_num, 5)}')
print(f'b_n 公式 = {np.round(b_formula, 5)}, 一致? {np.allclose(b_num, b_formula, atol=1e-4)}')

def square_partial_sum(t, n_terms):
    odd = np.arange(1, 2 * n_terms, 2)
    return 4 / np.pi * np.sum(np.sin(np.outer(t, odd) * w0) / odd, axis=1)

t = np.linspace(0, T, 20001)
plt.figure(figsize=(4, 3))
plt.plot(t, square(t), 'k', linewidth=1, label='square wave')
for n_terms in [1, 3, 10, 50]:
    partial = square_partial_sum(t, n_terms)
    plt.plot(t, partial, label=f'{n_terms} terms')
    print(f'{n_terms:>3} 项: 最大值 = {partial.max():.5f}, 均方误差 = {np.mean((partial - square(t))**2):.5f}')
plt.xlabel('t')
plot_nicely()
plt.show()
print('Gibbs 极限 (2/pi) Si(pi) =', round(2 / np.pi * quad(lambda x: np.sin(x) / x, 0, np.pi)[0], 5))

<a id="complex-fourier"></a>

$$
\text{复数形式 (Complex Form)} \\
\text{由欧拉公式 } \cos x = \frac{e^{jx} + e^{-jx}}{2}, \ \sin x = \frac{e^{jx} - e^{-jx}}{2j} \text{, 代入实数形式:} \\
a_n \cos n\omega_0 t + b_n \sin n\omega_0 t = \underbrace{\frac{a_n - j b_n}{2}}_{c_n} e^{jn\omega_0 t} + \underbrace{\frac{a_n + j b_n}{2}}_{c_{-n}} e^{-jn\omega_0 t} \\
f(t) = \sum_{n=-\infty}^{\infty} c_n e^{jn\omega_0 t}, \quad c_n = \frac{1}{T}\int_0^T f(t) e^{-jn\omega_0 t} \, dt \\
\text{(复指数同样正交: } \int_0^T e^{jm\omega_0 t} e^{-jn\omega_0 t} dt = \int_0^T e^{j(m-n)\omega_0 t} dt = T \delta_{mn} \text{, 求系数时乘的是共轭 } e^{-jn\omega_0 t} \text{)} \\
\text{对实函数 } c_{-n} = c_n^* \text{, 第 n 次谐波的振幅 } = \sqrt{a_n^2 + b_n^2} = 2|c_n| \\ \quad \\
\text{帕塞瓦尔定理 (Parseval's Theorem): 时域的平均功率 = 各频率分量的功率之和} \\
\frac{1}{T}\int_0^T |f|^2 dt = \frac{1}{T}\int_0^T \sum_m c_m e^{jm\omega_0 t} \sum_n c_n^* e^{-jn\omega_0 t} dt = \sum_m \sum_n c_m c_n^* \delta_{mn} = \sum_n |c_n|^2 \\
\text{方波: 左边 = 1, 右边 } = \sum_{\text{n 奇数}} 2 \cdot \frac{1}{4}\left(\frac{4}{n\pi}\right)^2 = \frac{8}{\pi^2}\left(1 + \frac{1}{9} + \frac{1}{25} + \cdots\right) \implies \sum_{\text{n 奇数}} \frac{1}{n^2} = \frac{\pi^2}{8} \\
\text{(直流 + 交流的 RMS 公式 } V_{rms}^2 = V_{dc}^2 + \frac{A^2}{2} \text{ 是它的特例)}
$$

数学基础: [欧拉公式](../math_basics/calculus.ipynb#taylor) · 应用: [直流 + 交流的有效值](AC_DC.ipynb#dc-offset)

In [ ]:
# 复系数 c_n (梯形法), 和 (a_n - j b_n) / 2 对比
t_int = np.linspace(0, T, 200001)
c = np.array([np.trapezoid(square(t_int) * np.exp(-1j * k * w0 * t_int), t_int) / T for k in range(-5, 6)])
expected = np.array([(a_num[abs(k)] - 1j * np.sign(k) * b_num[abs(k)]) / 2 if k != 0 else a_num[0] / 2 for k in range(-5, 6)])
print(f'c_n (n = -5..5) 虚部 = {np.round(c.imag, 4)}')
print(f'c_n == (a_n - j b_n)/2 ? {np.allclose(c, expected, atol=1e-6)}, c_(-n) == conj(c_n) ? {np.allclose(c[::-1], np.conj(c))}')

odd = np.arange(1, 200001, 2)
print(f'帕塞瓦尔: 左边 = 1, 右边 (前 10 万个奇数项) = {np.sum(8 / (np.pi**2 * odd**2)):.6f}, sum 1/n^2 (奇数) = {np.sum(1 / odd**2):.6f}, pi^2/8 = {np.pi**2 / 8:.6f}')

<a id="dft"></a>

$$
\text{离散傅里叶变换 (Discrete Fourier Transform, DFT)} \\
\text{在一个周期内等间隔采样 N 个点 } x_k = f(k \Delta t), \ \Delta t = \frac{T}{N} \text{, 用左黎曼和近似 } c_n \text{ 的积分:} \\
c_n = \frac{1}{T}\int_0^T f(t) e^{-jn\omega_0 t} dt \approx \frac{1}{T}\sum_{k=0}^{N-1} x_k e^{-j n \frac{2\pi}{T} k \frac{T}{N}} \frac{T}{N} = \frac{1}{N}\sum_{k=0}^{N-1} x_k e^{-j 2\pi nk / N} \\
\text{定义 DFT (numpy 的约定, 不除以 N): } X_n = \sum_{k=0}^{N-1} x_k e^{-j 2\pi nk / N}, \ n = 0, \cdots, N - 1 \implies c_n \approx \frac{X_n}{N} \\ \quad \\
\text{写成矩阵乘法 } X = F x \text{, } F_{nk} = W^{nk}, \ W = e^{-j 2\pi / N} \text{ (DFT 矩阵, 对称矩阵)} \\
\text{离散正交性 (等比数列求和, } W^{m-n} \neq 1 \text{ 时):} \sum_{k=0}^{N-1} W^{(m-n)k} = \frac{1 - W^{(m-n)N}}{1 - W^{m-n}} = \frac{1 - e^{-j2\pi(m-n)}}{1 - W^{m-n}} = 0 \\
\implies F F^H = N I \text{ (} F^H \text{ 是共轭转置), } \frac{F}{\sqrt{N}} \text{ 是酉矩阵 (Unitary Matrix, 复数版的正交矩阵)} \\
\text{逆变换 (IDFT): } x = F^{-1} X = \frac{1}{N} F^H X \implies x_k = \frac{1}{N}\sum_{n=0}^{N-1} X_n e^{j 2\pi nk / N} \\ \quad \\
\text{直接矩阵乘法 } O(N^2) \text{, 快速傅里叶变换 (FFT, Cooley-Tukey) 利用 } W^{nk} \text{ 的对称性分治, } O(N \log N)
$$

数学基础: [矩阵乘法](../math_basics/linear_algebra.ipynb#matrix-multiplication) · [正交矩阵](../math_basics/linear_algebra.ipynb#orthogonal) · [对称矩阵](../math_basics/linear_algebra.ipynb#symmetric-matrix)

In [ ]:
def dft_matrix(N):
    k = np.arange(N)
    return np.exp(-2j * np.pi * np.outer(k, k) / N)

N = 64
F = dft_matrix(N)
rng = np.random.default_rng(0)
x = rng.normal(size=N)
X_manual = F @ x
print(f'F x == np.fft.fft(x) ? {np.allclose(X_manual, np.fft.fft(x))}')
print(f'F F^H == N I ? {np.allclose(F @ F.conj().T, N * np.eye(N))}, F 对称? {np.allclose(F, F.T)}')
print(f'逆变换 (1/N) F^H X == x ? {np.allclose(F.conj().T @ X_manual / N, x)}, == np.fft.ifft ? {np.allclose(np.fft.ifft(X_manual), x)}')

# DFT 近似傅里叶系数: 方波采样后 X_n / N vs c_n
N_sq = 1024
x_sq = square(np.arange(N_sq) * T / N_sq)
X_sq = np.fft.fft(x_sq) / N_sq
print(f'方波: 2|X_n|/N (n = 1, 3, 5) = {np.round(2 * np.abs(X_sq[[1, 3, 5]]), 5)}, 4/(n pi) = {np.round(4 / (np.pi * np.array([1, 3, 5])), 5)}')

# 速度: O(N^2) vs O(N log N)
N_big = 2048
x_big = rng.normal(size=N_big)
F_big = dft_matrix(N_big)
t0 = time.perf_counter(); F_big @ x_big; t_mat = time.perf_counter() - t0
t0 = time.perf_counter(); np.fft.fft(x_big); t_fft = time.perf_counter() - t0
print(f'N = {N_big}: 矩阵乘法 {t_mat * 1000:.2f} ms, FFT {t_fft * 1000:.3f} ms')

<a id="frequency-resolution"></a>

$$
\text{DFT 的频率轴与频率分辨率 (Frequency Resolution)} \\
\text{采样频率 } f_s = \frac{1}{\Delta t} \text{, 采样 N 个点, 记录时长 } T = N \Delta t \text{, 第 n 个点对应频率 } f_n = \frac{n}{T} = n \frac{f_s}{N} \\
\Delta f = \frac{f_s}{N} = \frac{1}{T} \text{: 只和记录时长有关, 要分辨相差 0.5 Hz 的两个频率, 至少要记录 2 秒} \\
\text{实信号 } X_{N-n} = X_n^* \text{ (共轭对称), 只需看 } 0 \leq n \leq N/2 \text{, 单边振幅谱 } A_n = \frac{2|X_n|}{N} \text{ (n = 0 时不乘 2)} \\
\text{当信号频率不是 } \Delta f \text{ 的整数倍时, 能量会泄漏到相邻频点 (频谱泄漏 Spectral Leakage)}
$$

In [ ]:
fs = 100.0
fig, axs = plt.subplots(1, 2, figsize=(8, 3))
for ax, T_rec in zip(axs, [1.0, 4.0]):
    t_rec = np.arange(0, T_rec, 1 / fs)
    x_rec = np.sin(2 * np.pi * 10 * t_rec) + 0.8 * np.sin(2 * np.pi * 10.5 * t_rec)
    X = np.fft.rfft(x_rec)
    freqs = np.fft.rfftfreq(len(x_rec), 1 / fs)
    ax.stem(freqs, 2 * np.abs(X) / len(x_rec), basefmt=' ')
    ax.set_xlim(5, 15)
    ax.set_title(f'T = {T_rec} s, df = {freqs[1]:.2f} Hz')
    ax.set_xlabel('f (Hz)')
    print(f'记录 {T_rec} s: 频率分辨率 = 1/T = {1 / T_rec:.2f} Hz, numpy 频率间隔 = {freqs[1]:.2f} Hz')
plot_nicely()
plt.show()

<a id="aliasing"></a>

$$
\text{采样定理 (Nyquist-Shannon Sampling Theorem) 与混叠 (Aliasing)} \\
\text{以 } f_s \text{ 采样 } \cos(2\pi f t) \text{, 第 k 个样本 } x_k = \cos\left(2\pi \frac{f}{f_s} k\right) \\
\text{对任意整数 m, } \cos\left(2\pi \frac{f - m f_s}{f_s} k\right) = \cos\left(2\pi \frac{f}{f_s} k - 2\pi m k\right) = x_k \text{ (mk 是整数)} \\
\text{又因为 cos 是偶函数, } m f_s - f \text{ 也得到同样的样本} \\
\implies \text{频率 } f, \ |f - f_s|, \ |f - 2f_s|, \cdots \text{ 采样后无法区分, 都会显示成 } [0, \tfrac{f_s}{2}] \text{ 内的那一个} \\
\text{要无失真地采样, 信号最高频率必须 } f_{max} < \frac{f_s}{2} \text{ (奈奎斯特频率 Nyquist Frequency)} \\
\text{例: 9 Hz 的信号用 10 Hz 采样, 看起来像 1 Hz (车轮在视频里倒转也是这个原因); 所以 ADC 前面要加低通 (抗混叠) 滤波器}
$$

In [ ]:
fs_low = 10.0
t_cont = np.linspace(0, 2, 4000)
t_samp = np.arange(0, 2, 1 / fs_low)
f_true = 9.0
alias = abs(f_true - round(f_true / fs_low) * fs_low)
print(f'{f_true} Hz 以 {fs_low} Hz 采样 -> 混叠频率 = {alias} Hz')
print(f'样本完全相同? {np.allclose(np.cos(2 * np.pi * f_true * t_samp), np.cos(2 * np.pi * alias * t_samp))}')

for f_test in [3.0, 6.0, 9.0, 13.0, 21.0]:
    x_s = np.cos(2 * np.pi * f_test * np.arange(0, 10, 1 / fs_low))
    freqs = np.fft.rfftfreq(len(x_s), 1 / fs_low)
    f_peak = freqs[np.argmax(np.abs(np.fft.rfft(x_s)))]
    f_pred = abs(f_test - round(f_test / fs_low) * fs_low)
    print(f'真实 {f_test:>4} Hz -> FFT 峰值 {f_peak:.1f} Hz, 推导 |f - round(f/fs) fs| = {f_pred:.1f} Hz')

plt.figure(figsize=(4, 3))
plt.plot(t_cont, np.cos(2 * np.pi * f_true * t_cont), alpha=0.5, label='9 Hz signal')
plt.plot(t_cont, np.cos(2 * np.pi * alias * t_cont), 'r--', label='1 Hz alias')
plt.plot(t_samp, np.cos(2 * np.pi * f_true * t_samp), 'ko', label='samples (fs = 10 Hz)')
plt.xlabel('t (s)')
plot_nicely()
plt.show()

<a id="harmonics"></a>

$$
\text{应用: 交流电的谐波分析 (Harmonic Analysis)} \\
\text{非线性负载 (整流器, 开关电源) 让电流不再是纯正弦, 含有 3, 5, 7 次等奇次谐波 (奇对称波形 } f(t + T/2) = -f(t) \text{ 没有偶次谐波)} \\
\text{总谐波失真 (Total Harmonic Distortion): } THD = \frac{\sqrt{V_2^2 + V_3^2 + \cdots}}{V_1} \text{ (各次谐波有效值, 由帕塞瓦尔定理)} \\
\text{总有效值 } V_{rms} = \sqrt{V_1^2 + V_2^2 + \cdots} = V_1 \sqrt{1 + THD^2}
$$

In [ ]:
f_line, fs_line = 50.0, 10000.0
t_line = np.arange(0, 0.2, 1 / fs_line)          # 10 个完整周期, 50 Hz 正好落在频点上, 没有泄漏
amps = {1: 311.0, 3: 40.0, 5: 25.0, 7: 10.0}
phases = {1: 0.0, 3: 0.4, 5: -1.0, 7: 2.0}
v_line = sum(A * np.sin(2 * np.pi * h * f_line * t_line + phases[h]) for h, A in amps.items())

X_line = np.fft.rfft(v_line)
freqs = np.fft.rfftfreq(len(v_line), 1 / fs_line)
amp_spec = 2 * np.abs(X_line) / len(v_line)
for h in amps:
    k = np.argmin(np.abs(freqs - h * f_line))
    print(f'{h} 次谐波 ({freqs[k]:.0f} Hz): FFT 振幅 = {amp_spec[k]:.4f}, 真值 = {amps[h]}')

V_rms_h = np.array([amps[h] for h in amps]) / np.sqrt(2)
thd = np.sqrt(np.sum(V_rms_h[1:]**2)) / V_rms_h[0]
print(f'THD = {thd * 100:.2f}%')
print(f'有效值: 时域 = {np.sqrt(np.mean(v_line**2)):.4f} V, V1 sqrt(1 + THD^2) = {V_rms_h[0] * np.sqrt(1 + thd**2):.4f} V')

fig, axs = plt.subplots(1, 2, figsize=(8, 3))
axs[0].plot(t_line[:400] * 1000, v_line[:400], label='distorted voltage')
axs[0].plot(t_line[:400] * 1000, 311 * np.sin(2 * np.pi * f_line * t_line[:400]), '--', label='fundamental')
axs[0].set_xlabel('t (ms)')
axs[1].stem(freqs[:80], amp_spec[:80], basefmt=' ')
axs[1].set_xlabel('f (Hz)')
axs[1].set_title('amplitude spectrum')
plot_nicely()
plt.show()